# Snake: from Q-table to DQN 🐍

You will build a Snake game from scratch, then teach a computer to play it – first with a **lookup table** (tabular Q-learning), then with a **neural network** (DQN) – and finally race the agents against each other.

| Milestone | What you build | You learn |
|---|---|---|
| 1 | The game as a headless, gym-like environment + a random agent | what an RL environment is: `reset()`, `step(action) → state, reward, done` |
| 2 | Tabular Q-learning on an 11-feature compact state | Q-tables, ε-greedy exploration, the TD update |
| 3 | DQN in PyTorch | replay buffer, target network, ε-schedule |
| 4 | Race: random vs. Q-table vs. DQN + watch the winner play | learning curves, why greedy evaluation ≠ training score |

**How it works.** Cells marked `# TODO` are yours. Each has a ✅/❌ check right below it and a hidden solution (click ▶ *Solution*). The notebook runs top to bottom even before you solve anything – the agents just won't learn until you do.

**Runtime:** about 1–2 minutes on a laptop CPU in total. **Needs:** `numpy`, `matplotlib`, `torch`.

Theory: [[RL Basics and MDPs]] · [[Q-Learning and Policy Gradients]]. Sequel: [RL Flappy Bird - Q-Table to DQN](RL%20Flappy%20Bird%20-%20Q-Table%20to%20DQN.ipynb).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, copy, time
from collections import deque
import torch, torch.nn as nn
from matplotlib import animation
from IPython.display import HTML, display

torch.set_num_threads(1)   # tiny networks are fastest on one thread (and runs are reproducible)
def check(ok, msg_ok, msg_bad):
    print(("✅ " + msg_ok) if ok else ("❌ " + msg_bad))
def rolling(x, w=50):
    x = np.asarray(x, float); c = np.cumsum(np.insert(x, 0, 0))
    return np.array([(c[i + 1] - c[max(0, i + 1 - w)]) / min(i + 1, w) for i in range(len(x))])

---
## Milestone 1 – The game
**Rules.** A 10×10 board. The snake starts with length 3 in the middle, heading right. Each step it moves one cell. Eating food (+1 score) makes it one cell longer and new food appears on a random free cell. Hitting a wall or its own body ends the game.

**The RL interface** (the same shape as OpenAI Gym / Gymnasium):
- `state = env.reset()` starts a new game.
- `state, reward, done = env.step(action)` advances one step.

**Design decisions** (each one matters for learning):
- **Relative actions:** `0` = straight, `1` = turn right, `2` = turn left. With absolute actions (up/down/left/right) one of the four would always be instant suicide (reversing into yourself), and the agent would have to learn the same thing four times, once per heading.
- **Rewards:** +10 for food, −10 for dying, 0 otherwise.
- **Starvation:** if the snake goes 400 steps without eating, the episode is cut off – otherwise an agent that learned to circle forever would never finish an episode.
- **Compact state – 11 yes/no features** instead of the full board:

| # | Feature | # | Feature |
|---|---|---|---|
| 0 | danger straight ahead | 7 | food is above the head |
| 1 | danger to the right | 8 | food is right of the head |
| 2 | danger to the left | 9 | food is below the head |
| 3–6 | heading up / right / down / left (one-hot) | 10 | food is left of the head |

"Danger" means the next cell in that direction is a wall or body. The agent never sees the whole board – so it cannot plan around its own tail. Keep that in mind when you watch it play.

In [ ]:
class Snake:
    """Headless Snake. Actions are relative to the heading: 0 = straight, 1 = turn right, 2 = turn left."""
    DIRS = [(-1, 0), (0, 1), (1, 0), (0, -1)]   # up, right, down, left as (d_row, d_col)
    FEATURES = ["danger ahead", "danger right", "danger left", "heading up", "heading right", "heading down",
                "heading left", "food up", "food right", "food down", "food left"]
    FOOD_REWARD, DEATH_REWARD, STEP_REWARD = 10.0, -10.0, 0.0

    def __init__(self, size=10, seed=None):
        self.size = size
        self.rng = np.random.default_rng(seed)
        self.max_hunger = 4 * size * size

    def reset(self):
        c = self.size // 2
        self.heading = 1                                       # start heading right
        self.body = deque([(c, c), (c, c - 1), (c, c - 2)])   # head first
        self.score, self.steps, self.hunger = 0, 0, 0
        self._place_food()
        return self.get_state()

    def _place_food(self):
        free = [(r, c) for r in range(self.size) for c in range(self.size) if (r, c) not in self.body]
        self.food = free[self.rng.integers(len(free))] if free else None

    def _hits(self, cell):
        """Would the head die in `cell`? The tail cell is safe: it moves away on the same step."""
        r, c = cell
        return not (0 <= r < self.size and 0 <= c < self.size) or cell in list(self.body)[:-1]

    def step(self, action):
        self.heading = (self.heading + (0, 1, -1)[action]) % 4
        dr, dc = self.DIRS[self.heading]
        head = (self.body[0][0] + dr, self.body[0][1] + dc)
        self.steps += 1; self.hunger += 1
        if self._hits(head):
            return self.get_state(), self.DEATH_REWARD, True
        self.body.appendleft(head)
        if head == self.food:
            self.score += 1; self.hunger = 0
            self._place_food()
            return self.get_state(), self.FOOD_REWARD, self.food is None   # board full = you won
        self.body.pop()                                                    # no food: the tail moves along
        return self.get_state(), self.STEP_REWARD, self.hunger >= self.max_hunger

    def get_state(self):
        hr, hc = self.body[0]
        def danger(turn):
            dr, dc = self.DIRS[(self.heading + turn) % 4]
            return self._hits((hr + dr, hc + dc))
        fr, fc = self.food if self.food else (hr, hc)
        return np.array([danger(0), danger(1), danger(-1),
                         self.heading == 0, self.heading == 1, self.heading == 2, self.heading == 3,
                         fr < hr, fc > hc, fr > hr, fc < hc], dtype=np.float32)

    def board(self):
        """Board as an int grid for drawing: 0 empty, 1 body, 2 head, 3 food."""
        g = np.zeros((self.size, self.size), int)
        for r, c in list(self.body)[1:]: g[r, c] = 1
        g[self.body[0]] = 2
        if self.food: g[self.food] = 3
        return g

env = Snake(seed=0)
state = env.reset()
for _ in range(3):
    state, reward, done = env.step(0)   # go straight three times
print("score", env.score, "| reward", reward, "| done", done)
print({name: int(v) for name, v in zip(Snake.FEATURES, state)})

In [ ]:
from matplotlib.colors import ListedColormap
CMAP = ListedColormap(["#f4f4f0", "#2f9e44", "#1b5e20", "#e8590c"])
def draw(env, ax=None, title=None):
    ax = ax or plt.gca()
    img = ax.imshow(env.board(), cmap=CMAP, vmin=0, vmax=3)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title or f"score {env.score}")
    return img

draw(env); plt.show()

### Baseline: a random agent
Before any learning, measure what *no* intelligence scores. Every agent below is evaluated the same way: **100 games on a fixed-seed board** (`seed=123`), always taking its best action (no exploration).

In [ ]:
def play(env, policy):
    """Play one game with policy(state) -> action; return the score."""
    state, done = env.reset(), False
    while not done:
        state, _, done = env.step(policy(state))
    return env.score

def evaluate(policy, n=100, seed=123):
    env = Snake(seed=seed)
    return np.array([play(env, policy) for _ in range(n)])

rng = np.random.default_rng(0)
random_scores = evaluate(lambda s: int(rng.integers(3)))
print(f"random agent: mean score {random_scores.mean():.2f}, best {random_scores.max()}")

---
## Milestone 2 – Tabular Q-learning
A **Q-table** has one row per state and one column per action. `Q[s, a]` estimates the total discounted reward you get by taking action `a` in state `s` and playing well afterwards. With 11 yes/no features there are at most $2^{11} = 2048$ states (many can never happen, e.g. two headings at once), so the table is only 2048 × 3.

Three pieces to write: turn a state into a row number, pick actions with ε-greedy, and update the table.

### Exercise 1 – state → row number
Read the 11 features as the bits of a binary number: feature `i` contributes $2^i$ if it is 1. So `[1,0,1,0,…,0]` → $1 + 4 = 5$.

In [ ]:
def state_to_index(state):
    # TODO: return an int in [0, 2048) – a different one for every different state
    return 0

<details><summary>▶ Solution</summary>

```python
def state_to_index(state):
    return int(state.astype(int) @ (1 << np.arange(11)))
```

`1 << np.arange(11)` is `[1, 2, 4, …, 1024]`; the dot product adds up the powers of two whose bit is set.
</details>

In [ ]:
all_states = ((np.arange(2048)[:, None] >> np.arange(11)) & 1).astype(np.float32)   # every possible 11-bit state
idx = [state_to_index(s) for s in all_states]
check(len(set(idx)) == 2048 and min(idx) >= 0 and max(idx) < 2048,
      "All 2048 states get their own row.", f"Only {len(set(idx))} distinct rows (need 2048, all in [0, 2048)).")

### Exercise 2 – ε-greedy
With probability ε pick a **random** action (explore), otherwise the action with the **highest Q-value** (exploit). Without exploration the agent keeps repeating its first lucky guess and never discovers anything better.

In [ ]:
def epsilon_greedy(q_values, eps, rng):
    # TODO: with probability eps return a random action index, otherwise the argmax of q_values
    return 0

<details><summary>▶ Solution</summary>

```python
def epsilon_greedy(q_values, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(len(q_values)))
    return int(np.argmax(q_values))
```


</details>

In [ ]:
r = np.random.default_rng(1); q = np.array([0.0, 5.0, 1.0])
greedy = [epsilon_greedy(q, 0.0, r) for _ in range(1000)]
mixed = np.bincount([epsilon_greedy(q, 0.3, r) for _ in range(6000)], minlength=3) / 6000
print("action frequencies with eps=0.3:", mixed.round(3), "(expected [0.1, 0.8, 0.1])")
check(set(greedy) == {1} and abs(mixed[1] - 0.8) < 0.03 and min(mixed[0], mixed[2]) > 0.07,
      "Greedy picks the best action, ε=0.3 explores ~30 % of the time.",
      "eps=0 must always return the argmax; with eps=0.3 the best action should come up ~80 % (70 % greedy + 10 % random).")

### Exercise 3 – the Q-learning update
After a step $(s, a, r, s')$, move $Q(s,a)$ a fraction $\alpha$ towards the **TD target**:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\big[\underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{TD target}} - Q(s,a)\big]$$

If the episode ended (`done`), there is no future: the target is just $r$.

In [ ]:
def q_update(Q, s, a, r, s2, done, alpha, gamma):
    # TODO: update Q[s, a] in place using the rule above (s and s2 are row numbers)
    pass

<details><summary>▶ Solution</summary>

```python
def q_update(Q, s, a, r, s2, done, alpha, gamma):
    target = r if done else r + gamma * Q[s2].max()
    Q[s, a] += alpha * (target - Q[s, a])
```

Using `max` over the next state's actions (not the action you will actually take) is what makes Q-learning **off-policy**: it learns the value of the greedy policy while behaving ε-greedily.
</details>

In [ ]:
Qt = np.zeros((3, 2)); Qt[1] = [2.0, 4.0]
q_update(Qt, 0, 1, 1.0, 1, False, alpha=0.5, gamma=0.9)   # target 1 + 0.9*4 = 4.6 → Q = 0 + 0.5*4.6
q_update(Qt, 2, 0, -10.0, 1, True, alpha=0.5, gamma=0.9)  # terminal: target −10 → Q = −5
check(np.isclose(Qt[0, 1], 2.3) and np.isclose(Qt[2, 0], -5.0) and Qt[1].tolist() == [2.0, 4.0],
      "Correct TD update, including the terminal case.", f"Got Q[0,1]={Qt[0,1]:.2f} (want 2.3) and Q[2,0]={Qt[2,0]:.2f} (want −5.0).")

### Train it
The training loop just glues your three pieces together. ε starts at 1 (pure exploration) and is multiplied by 0.995 after every game, so after ~900 games the agent mostly exploits (ε bottoms out at 0.01).

In [ ]:
def train_tabular(episodes=1500, alpha=0.1, gamma=0.9, eps_decay=0.995, eps_min=0.01, seed=0):
    env, rng = Snake(seed=seed), np.random.default_rng(seed)
    Q, eps, scores = np.zeros((2048, 3)), 1.0, []
    for ep in range(episodes):
        s, done = state_to_index(env.reset()), False
        while not done:
            a = epsilon_greedy(Q[s], eps, rng)
            state2, r, done = env.step(a)
            s2 = state_to_index(state2)
            q_update(Q, s, a, r, s2, done, alpha, gamma)
            s = s2
        eps = max(eps_min, eps * eps_decay)
        scores.append(env.score)
    return Q, scores

t0 = time.time()
Q, tab_curve = train_tabular()
tab_policy = lambda s: int(np.argmax(Q[state_to_index(s)]))
tab_scores = evaluate(tab_policy)
print(f"trained in {time.time() - t0:.0f}s | rows ever visited: {(Q != 0).any(1).sum()} of 2048")
print(f"Q-table agent: mean score {tab_scores.mean():.2f} ± {tab_scores.std():.2f}, best {tab_scores.max()}")
plt.plot(tab_curve, alpha=.25, label="score per episode"); plt.plot(rolling(tab_curve), label="rolling mean (50)")
plt.xlabel("episode"); plt.ylabel("score"); plt.title("Tabular Q-learning"); plt.legend(); plt.show()
check(tab_scores.mean() >= 10, "The Q-table agent averages 10+ food per game.",
      "Below 10 – finish Exercises 1–3 first; all three checks must be ✅.")

### Peek inside the table
A Q-table is fully interpretable: you can ask it what it would do in any situation. Below are a few hand-picked states. Does the policy make sense?

In [ ]:
def describe(bits):
    s = np.zeros(11, np.float32); s[list(bits)] = 1
    q = Q[state_to_index(s)]
    best = ["straight", "right", "left"][int(np.argmax(q))]
    print(f"{', '.join(Snake.FEATURES[i] for i in bits):42s} → Q = {np.round(q, 1)} → go {best}")

describe([4, 8])        # heading right, food straight ahead (to the right)
describe([4, 9])        # heading right, food below → expect "right" (turning right from heading right = down)
describe([0, 4, 8])     # wall/body ahead, food to the right
describe([0, 1, 4, 9])  # danger ahead and right, food below

---
## Milestone 3 – Deep Q-Network (DQN)
A table needs one row per state. That is fine for 2048 states, hopeless for raw pixels. A **DQN** replaces the table with a small neural network $Q_\theta(s, \cdot)$ that outputs one Q-value per action, so similar states share what they learned. Three tricks make it train stably ([Mnih et al. 2015](https://www.nature.com/articles/nature14236)):

1. **Replay buffer** – store transitions and train on random mini-batches of old ones. Consecutive steps are highly correlated; shuffling them makes the gradient steps look more like ordinary supervised learning.
2. **Target network** – compute TD targets with a frozen copy $Q_{\theta^-}$ that is only synced every 1000 steps. Otherwise every update moves the target you are chasing.
3. **ε-schedule** – ε falls linearly from 1 to 0.01 over the first 10 000 steps.

The loss for a mini-batch is the Huber (smooth L1) loss between $Q_\theta(s,a)$ and the target $y = r + \gamma (1 - \text{done}) \max_{a'} Q_{\theta^-}(s', a')$.

In [ ]:
class ReplayBuffer:
    """Fixed-size ring buffer of transitions (s, a, r, s', done)."""
    def __init__(self, capacity, state_dim, rng):
        self.s, self.s2 = np.zeros((capacity, state_dim), np.float32), np.zeros((capacity, state_dim), np.float32)
        self.a, self.r, self.d = np.zeros(capacity, np.int64), np.zeros(capacity, np.float32), np.zeros(capacity, np.float32)
        self.capacity, self.size, self.pos, self.rng = capacity, 0, 0, rng
    def add(self, s, a, r, s2, done):
        i = self.pos
        self.s[i], self.a[i], self.r[i], self.s2[i], self.d[i] = s, a, r, s2, done
        self.pos = (i + 1) % self.capacity; self.size = min(self.size + 1, self.capacity)
    def sample(self, batch_size):
        j = self.rng.integers(self.size, size=batch_size)
        return tuple(torch.as_tensor(x[j]) for x in (self.s, self.a, self.r, self.s2, self.d))

def make_qnet(state_dim, n_actions, hidden=64):
    return nn.Sequential(nn.Linear(state_dim, hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_actions))

### Exercise 4 – TD targets for a batch
Given tensors `r` (rewards), `s2` (next states, shape `[B, 11]`) and `done` (1.0 if the episode ended, else 0.0), return the targets $y = r + \gamma (1 - \text{done}) \max_{a'} Q_{\theta^-}(s', a')$ as a tensor of shape `[B]`. Targets are constants: no gradients should flow through them.

In [ ]:
def td_targets(r, s2, done, target_net, gamma):
    # TODO: return r + gamma * (1 - done) * (max over actions of target_net(s2)), without gradients
    return torch.zeros_like(r)

<details><summary>▶ Solution</summary>

```python
def td_targets(r, s2, done, target_net, gamma):
    with torch.no_grad():
        return r + gamma * (1 - done) * target_net(s2).max(dim=1).values
```

`(1 - done)` switches the bootstrap term off at the end of an episode – exactly the `if done` branch of Exercise 3, but vectorised. `torch.no_grad()` keeps the target fixed while we take a gradient step on the online network.
</details>

In [ ]:
toy = nn.Linear(2, 3, bias=False)
with torch.no_grad(): toy.weight.copy_(torch.tensor([[1., 0.], [0., 1.], [1., 1.]]))   # Q(s') = [x, y, x + y]
y = td_targets(torch.tensor([1., 2.]), torch.tensor([[1., 2.], [5., -9.]]), torch.tensor([0., 1.]), toy, 0.5)
check(torch.allclose(y, torch.tensor([2.5, 2.0])) and not y.requires_grad,
      "Targets are right and carry no gradient.", f"Got {y.tolist()} (want [2.5, 2.0], requires_grad=False).")

### Train the DQN
The loop below takes one gradient step every 4 environment steps and, every 50 games, checks how well the **greedy** policy plays (5 test games). It keeps the best snapshot – DQN training is noisy and the last network is not always the best one.

In [ ]:
def train_dqn(total_steps=60_000, gamma=0.9, lr=1e-3, batch_size=64, eps_steps=10_000, target_sync=1000,
              train_every=4, warmup=1000, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    env, test_env = Snake(seed=seed), Snake(seed=999)
    net, target = make_qnet(11, 3), make_qnet(11, 3)
    target.load_state_dict(net.state_dict())
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    buf = ReplayBuffer(50_000, 11, rng)
    step, scores, best = 0, [], (-1.0, None)
    while step < total_steps:
        s, done = env.reset(), False
        while not done:
            eps = max(0.01, 1 - step / eps_steps)
            with torch.no_grad():
                a = epsilon_greedy(net(torch.as_tensor(s)).numpy(), eps, rng)
            s2, r, done = env.step(a)
            buf.add(s, a, r, s2, float(done)); s = s2; step += 1
            if buf.size >= warmup and step % train_every == 0:
                bs, ba, br, bs2, bd = buf.sample(batch_size)
                q = net(bs).gather(1, ba[:, None]).squeeze(1)              # Q(s, a) for the actions taken
                loss = nn.functional.smooth_l1_loss(q, td_targets(br, bs2, bd, target, gamma))
                opt.zero_grad(); loss.backward(); opt.step()
            if step % target_sync == 0:
                target.load_state_dict(net.state_dict())
        scores.append(env.score)
        if len(scores) % 50 == 0:
            with torch.no_grad():
                test = np.mean([play(test_env, lambda s: int(net(torch.as_tensor(s)).argmax())) for _ in range(5)])
            if test >= best[0]:
                best = (test, copy.deepcopy(net.state_dict()))
    net.load_state_dict(best[1])
    return net, scores

t0 = time.time()
qnet, dqn_curve = train_dqn()
def dqn_policy(s):
    with torch.no_grad():
        return int(qnet(torch.as_tensor(s)).argmax())
dqn_scores = evaluate(dqn_policy)
print(f"trained in {time.time() - t0:.0f}s on {len(dqn_curve)} games")
print(f"DQN agent: mean score {dqn_scores.mean():.2f} ± {dqn_scores.std():.2f}, best {dqn_scores.max()}")
check(dqn_scores.mean() >= 10, "The DQN agent averages 10+ food per game.",
      "Below 10 – finish Exercise 4 (and 2, the loop uses epsilon_greedy).")

---
## Milestone 4 – The race
Same 100 test games for everybody. Left: learning curves (score per training game, rolling mean of 50). Right: greedy test scores.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(rolling(tab_curve), label="Q-table"); a1.plot(rolling(dqn_curve), label="DQN")
a1.axhline(random_scores.mean(), color="gray", ls="--", label="random")
a1.set_xlabel("training game"); a1.set_ylabel("score (rolling mean 50)"); a1.set_title("Learning curves"); a1.legend()
names, results = ["random", "Q-table", "DQN"], [random_scores, tab_scores, dqn_scores]
a2.bar(names, [x.mean() for x in results], yerr=[x.std() for x in results], capsize=6, color=["#adb5bd", "#4c6ef5", "#e8590c"])
a2.set_ylabel("mean score over 100 test games (± std)"); a2.set_title("Greedy evaluation")
plt.tight_layout(); plt.show()
for n, x in zip(names, results):
    print(f"{n:8s} mean {x.mean():5.2f}  median {np.median(x):4.0f}  best {x.max():3d}")

**What to notice**
- Both learners crush the random agent. With the 11-feature state the **Q-table ends up within a few points of the DQN** (across seeds 0–2: Q-table 17–19, DQN 19–21) and trains in a fraction of the wall-clock time: the state space is tiny (only a few hundred rows are ever visited), so there is little for a network to generalise. Deep RL pays off when the state is too large for a table.
- The learning curves sit *below* the test scores: during training the agent still explores (ε > 0) and every random move can be fatal.
- Both agents plateau at roughly 15–25. The reason is the state, not the algorithm: the snake sees only the three cells next to its head, so it happily curls into a dead end formed by its own body. Watch it happen below.

<details><summary>▶ Think about it: the DQN needs fewer <i>games</i> to get good, but more <i>seconds</i>. Why?</summary>

- **Replay:** the DQN draws 64 transitions every 4 steps, so each transition is used about 16 times. The table learns from each transition exactly once.
- **Generalisation:** an update for one state also nudges the Q-values of similar states. The table must visit every row on its own.
- **Exploration schedule:** the DQN's ε reaches 0.01 after 10 000 steps (a few hundred games); the table's ε is still about 0.1 after 450 games ($0.995^{450}$).

Every one of those steps costs a forward and a backward pass through the network though, so in wall-clock time the table wins.
</details>

### Watch the winner play

In [ ]:
def record(policy, seed=7, max_frames=300):
    env = Snake(seed=seed); s, done, frames = env.reset(), False, [env.board()]
    while not done and len(frames) < max_frames:
        s, _, done = env.step(policy(s)); frames.append(env.board())
    return frames, env.score

best_name, best_policy = max([("Q-table", tab_policy), ("DQN", dqn_policy)], key=lambda p: evaluate(p[1], 20).mean())
frames, final = record(best_policy)
fig, ax = plt.subplots(figsize=(4, 4)); img = ax.imshow(frames[0], cmap=CMAP, vmin=0, vmax=3)
ax.set_xticks([]); ax.set_yticks([])
def update(i):
    img.set_data(frames[i]); ax.set_title(f"{best_name} – step {i}"); return [img]
anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=80)
plt.close(fig)
print(f"{best_name} scored {final} in this game ({len(frames) - 1} steps shown)")
HTML(anim.to_jshtml())

---
## Go further
- **Bigger board:** set `Snake(size=15)`. The Q-table does not grow (the state is still 11 bits) – does the score scale with the board?
- **See more:** give the DQN a 5×5 window around the head (25 extra inputs) or the whole board. A table can no longer cope ($2^{25}$+ rows), a network can. Does it stop trapping itself?
- **Double DQN:** pick $a' = \arg\max Q_\theta(s')$ with the online net but evaluate it with the target net. One-line change in `td_targets`.
- **Reward shaping:** add +0.1 for moving towards the food and −0.1 for moving away. Faster learning? Any weird behaviour?
- **Next game:** [RL Flappy Bird - Q-Table to DQN](RL%20Flappy%20Bird%20-%20Q-Table%20to%20DQN.ipynb) – continuous state, delayed rewards and a reward-design surprise. Or watch a Q-table learn live in [RL Arcade.html](RL%20Arcade.html).